# SalesMesh: Explainability of the Lead Score

Okeyo Hoppe Solini Marie (161031) · Supervisor: Mr. Tiberius Tabulu · Strathmore University

This notebook explains **why** the trained Gradient Boosting model gives each lead its score, at two levels:

1. **Global explanations**: which lead details matter most to the model overall
   (built-in feature importance, permutation importance and SHAP).
2. **Local explanations**: why one particular lead received its score. This uses the same
   "what-if" method that the SalesMesh application shows to marketers ("Time on website (1234) raises
   the score by 49 points"), and the result is checked against SHAP.

It uses the saved model `salesmesh_model.pkl`, so the explanations are for exactly the model used by the app.

## 1. Setup
The model was trained with scikit-learn 1.6.1 and imbalanced-learn 0.14.2, so the same versions are installed
here to load it correctly. **If Colab shows a "Restart session" button after this cell, click it, then run the
notebook again from the next cell.**

In [ ]:
!pip install -q scikit-learn==1.6.1 imbalanced-learn==0.14.2 shap

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import shap
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

RANDOM_STATE = 42
print("scikit-learn", sklearn.__version__, "| shap", shap.__version__)

## 2. Load the model, the cleaned leads and the reference values
Upload these three files from the project folder when asked:
- `salesmesh_model.pkl` (app/model)
- `reference_values.json` (app/model): the typical value of each detail in the training data
- `leads_cleaned.csv` (Data)

In [ ]:
try:
    from google.colab import files
    for name in ["salesmesh_model.pkl", "reference_values.json", "leads_cleaned.csv"]:
        print("Upload:", name)
        files.upload()
except ImportError:
    pass  # running outside Colab: the files are expected in the current folder

model = joblib.load("salesmesh_model.pkl")
reference = json.load(open("reference_values.json"))
df = pd.read_csv("leads_cleaned.csv")

FEATURES = list(model.named_steps["scaler"].feature_names_in_)   # the 132 encoded model features
INPUTS = [c for c in df.columns if c != "Converted"]               # the 14 lead details
NUMERIC = ["TotalVisits", "Total Time Spent on Website", "Page Views Per Visit"]
print(len(df), "leads |", len(INPUTS), "lead details |", len(FEATURES), "model features")
print(model)

## 3. Rebuild the test set and confirm it is the same model
The same 80/20 stratified split (random seed 42) as the training notebook is recreated. If the results match the
training notebook (F1 0.764, AUC-ROC 0.878), the explanations below describe the exact model that was evaluated.

In [ ]:
def encode(rows):
    """One-hot encode lead details and line the columns up with the model, exactly as the app does."""
    return pd.get_dummies(rows[INPUTS]).astype(float).reindex(columns=FEATURES, fill_value=0.0)

X_raw, y = df[INPUTS], df["Converted"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_test = encode(X_test_raw)

prob = model.predict_proba(X_test)[:, 1]
print(f"Test leads: {len(X_test)}")
print(f"F1-score:  {f1_score(y_test, (prob >= 0.5).astype(int)):.3f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, prob):.3f}")

## 4. Global explanation 1: the model's built-in feature importance
Gradient Boosting records how much each encoded feature reduced its prediction error while the trees were built.
This shows what the model relied on during training, at the level of single categories (for example one lead source).

In [ ]:
gb = model.named_steps["model"]
built_in = pd.Series(gb.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(built_in.head(10).round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
built_in.head(15).sort_values().plot.barh(ax=ax, color="#2a78d6")
ax.set_title("Gradient Boosting built-in feature importance (top 15)")
ax.set_xlabel("Importance")
plt.tight_layout(); plt.savefig("explain_builtin_importance.png", dpi=150); plt.show()

## 5. Global explanation 2: permutation importance by lead detail
Built-in importance splits each detail across many one-hot columns. Permutation importance is easier to read:
one lead detail at a time is shuffled across the test leads, and the fall in AUC-ROC is measured. The bigger the fall,
the more the model depends on that detail. Each detail is shuffled 5 times and the average is shown.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
base_auc = roc_auc_score(y_test, prob)
rows = []
for col in INPUTS:
    drops = []
    for _ in range(5):
        shuffled = X_test_raw.copy()
        shuffled[col] = rng.permutation(shuffled[col].values)
        drops.append(base_auc - roc_auc_score(y_test, model.predict_proba(encode(shuffled))[:, 1]))
    rows.append({"Lead detail": col, "AUC drop (mean)": np.mean(drops), "AUC drop (std)": np.std(drops)})
perm = pd.DataFrame(rows).sort_values("AUC drop (mean)", ascending=False).reset_index(drop=True)
display(perm.round(4))

fig, ax = plt.subplots(figsize=(9, 6))
p = perm.sort_values("AUC drop (mean)")
ax.barh(p["Lead detail"], p["AUC drop (mean)"], xerr=p["AUC drop (std)"], color="#2a78d6")
ax.set_title("Permutation importance on the test set")
ax.set_xlabel("Fall in AUC-ROC when the detail is shuffled")
plt.tight_layout(); plt.savefig("explain_permutation_importance.png", dpi=150); plt.show()

## 6. Global explanation 3: SHAP values
SHAP (SHapley Additive exPlanations) splits each prediction into the contribution of every feature, using
game theory. For tree models it is exact and fast. Contributions are in **log-odds**: positive values push a lead
towards converting, negative values push it away. The summary plot shows every test lead as one dot per feature;
red means a high value of that feature and blue a low value.

In [ ]:
X_test_scaled = pd.DataFrame(model.named_steps["scaler"].transform(X_test), columns=FEATURES, index=X_test.index)
explainer = shap.TreeExplainer(gb)
shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, X_test_scaled, max_display=15, show=False)
plt.title("SHAP summary (top 15 features)")
plt.tight_layout(); plt.savefig("explain_shap_summary.png", dpi=150); plt.show()

In [ ]:
# Combine the one-hot columns back into the 14 lead details (mean absolute SHAP value)
def detail_of(feature):
    for col in INPUTS:
        if feature == col or feature.startswith(col + "_"):
            return col
    return feature

abs_shap = pd.DataFrame(np.abs(shap_values), columns=FEATURES)
by_detail = abs_shap.T.groupby(detail_of).sum().T.mean().sort_values(ascending=False)
display(by_detail.round(3).rename("mean |SHAP| (log-odds)").to_frame())

compare = pd.DataFrame({"SHAP rank": by_detail.rank(ascending=False).astype(int),
                        "Permutation rank": perm.set_index("Lead detail")["AUC drop (mean)"]
                                                .rank(ascending=False).astype(int)})
print("Top 5 by SHAP:        ", list(by_detail.index[:5]))
print("Top 5 by permutation: ", list(perm["Lead detail"][:5]))

## 7. Local explanation: why one lead got its score (the method used in the app)
For a single lead, each detail is replaced in turn by the **typical value** from the training data (the median for
numbers, the most common value for categories), and the change in the score is measured. Details that move the score
most become the plain-language reasons the marketer sees. Effects under 2 points are not shown, and at most three
reasons are given. This is the same code as `explain()` in the SalesMesh application (`salesmesh/scoring.py`).

In [ ]:
FRIENDLY = {"Lead Origin": "Lead origin", "Lead Source": "Lead source", "Do Not Email": "Do-not-email request",
            "Do Not Call": "Do-not-call request", "TotalVisits": "Website visits",
            "Total Time Spent on Website": "Time on website", "Page Views Per Visit": "Pages viewed per visit",
            "Last Activity": "Last activity", "Country": "Country", "Specialization": "Specialisation",
            "How did you hear about X Education": "How they heard about us",
            "What is your current occupation": "Occupation", "City": "City",
            "Last Notable Activity": "Last notable activity"}
baseline = reference["baseline"]

def score(rows):
    return model.predict_proba(encode(rows))[:, 1]

def what_if_effects(lead):
    """Score change (in percentage points) caused by each detail that differs from the typical lead."""
    lead = pd.DataFrame([lead])[INPUTS]
    actual = score(lead)[0]
    effects = {}
    for col in INPUTS:
        same = (float(lead[col].iloc[0]) == float(baseline[col])) if col in NUMERIC else (str(lead[col].iloc[0]) == str(baseline[col]))
        if not same:
            typical = lead.copy(); typical[col] = baseline[col]
            effects[col] = (actual - score(typical)[0]) * 100
    return actual, pd.Series(effects, dtype=float).sort_values(key=abs, ascending=False)

def reasons(lead, top_n=3, min_points=2):
    actual, effects = what_if_effects(lead)
    out = []
    for col, pts in effects.head(top_n).items():
        if abs(pts) < min_points:
            continue
        value = f"({lead[col]:g})" if col in NUMERIC else f"'{lead[col]}'"
        out.append(f"{FRIENDLY[col]} {value} {'raises' if pts > 0 else 'lowers'} the score by {abs(pts):.0f} points")
    return actual, out

def tier(p):
    return "High" if p >= 0.70 else ("Medium" if p >= 0.40 else "Low")

In [ ]:
# The three example leads used in the app's Add leads page
examples = {
    "Hot lead":  {"Lead Origin": "Lead Add Form", "Lead Source": "Reference", "What is your current occupation": "Working Professional",
                  "TotalVisits": 4, "Total Time Spent on Website": 1100, "Page Views Per Visit": 3,
                  "Last Activity": "SMS Sent", "Last Notable Activity": "SMS Sent"},
    "Warm lead": {"Lead Origin": "Landing Page Submission", "Lead Source": "Direct Traffic", "What is your current occupation": "Unemployed",
                  "TotalVisits": 5, "Total Time Spent on Website": 700, "Page Views Per Visit": 3,
                  "Last Activity": "Email Opened", "Last Notable Activity": "Email Opened"},
    "Cold lead": {"Lead Origin": "Landing Page Submission", "Lead Source": "Google", "What is your current occupation": "Unemployed",
                  "TotalVisits": 2, "Total Time Spent on Website": 120, "Page Views Per Visit": 2,
                  "Last Activity": "Olark Chat Conversation", "Last Notable Activity": "Modified"},
}
for name, details in examples.items():
    lead = {**baseline, **details}
    p, why = reasons(lead)
    print(f"{name}: {p * 100:.0f}% chance of converting -> {tier(p)} priority")
    for r in why or ["Close to a typical lead: nothing stands out"]:
        print("   -", r)
    print()

## 8. Local explanation for real test leads
Three leads from the test set are explained: a High-priority lead that converted, a High-priority lead that did
**not** convert (a false positive) and a Low-priority lead that converted (a false negative). Looking at the errors
shows what the model cannot see.

In [ ]:
test = X_test_raw.copy()
test["prob"], test["Converted"] = prob, y_test.values
picks = {
    "High priority, converted": test[(test.prob >= 0.7) & (test.Converted == 1)].sort_values("prob").iloc[len(test[(test.prob >= 0.7) & (test.Converted == 1)]) // 2],
    "High priority, did NOT convert (false positive)": test[(test.prob >= 0.7) & (test.Converted == 0)].sort_values("prob", ascending=False).iloc[0],
    "Low priority, converted (false negative)": test[(test.prob < 0.4) & (test.Converted == 1)].sort_values("prob").iloc[0],
}
for title, row in picks.items():
    lead = row[INPUTS].to_dict()
    p, why = reasons(lead)
    print(f"{title}: score {p * 100:.0f}% ({tier(p)}), actual outcome: {'converted' if row.Converted else 'not converted'}")
    for r in why or ["Close to a typical lead: nothing stands out"]:
        print("   -", r)
    print()

## 9. Does the app's explanation agree with SHAP?
For the same leads, the details ranked highest by the app's what-if method are compared with the details ranked
highest by SHAP (SHAP values of each detail's one-hot columns added together). Agreement between two different
methods gives confidence that the reasons shown to marketers reflect what the model actually uses.

In [ ]:
def shap_by_detail(lead):
    x = pd.DataFrame(model.named_steps["scaler"].transform(encode(pd.DataFrame([lead]))), columns=FEATURES)
    sv = pd.Series(explainer.shap_values(x)[0], index=FEATURES)
    return sv.groupby(detail_of).sum().sort_values(key=abs, ascending=False)

rows = []
cases = {**{k: {**baseline, **v} for k, v in examples.items()}, **{k: v[INPUTS].to_dict() for k, v in picks.items()}}
for name, lead in cases.items():
    _, wi = what_if_effects(lead)
    sh = shap_by_detail(lead)
    top_wi, top_sh = list(wi.index[:3]), list(sh.index[:3])
    rows.append({"Lead": name, "Top 3 (app what-if)": ", ".join(FRIENDLY[c] for c in top_wi),
                 "Top 3 (SHAP)": ", ".join(FRIENDLY.get(c, c) for c in top_sh),
                 "Shared": len(set(top_wi) & set(top_sh)), "Same top detail": top_wi[0] == top_sh[0]})
agreement = pd.DataFrame(rows)
display(agreement)
print(f"On average {agreement['Shared'].mean():.1f} of the top 3 details are the same in both methods;")
print(f"the single most important detail is the same in {agreement['Same top detail'].sum()} of {len(agreement)} leads.")

In [ ]:
# A visual explanation of one lead: every detail's what-if effect
name = "Warm lead"
p, effects = what_if_effects({**baseline, **examples[name]})
e = effects.sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([FRIENDLY[c] for c in e.index], e.values, color=["#eb6834" if v < 0 else "#2a78d6" for v in e.values])
ax.axvline(0, color="#52514e", linewidth=0.8)
ax.set_title(f"{name}: {p * 100:.0f}% chance of converting ({tier(p)} priority)")
ax.set_xlabel("Change in score (percentage points) compared with a typical lead")
plt.tight_layout(); plt.savefig("explain_local_warm_lead.png", dpi=150); plt.show()

## 10. Summary
- **Globally**, permutation importance and SHAP both rank *time spent on the website* as the most important detail,
  followed by how the lead was acquired (lead origin), the lead's last activity and their occupation (sections 5 and 6).
- **Locally**, each score in SalesMesh is explained by changing one detail at a time to its typical value and showing
  the details that move the score most (sections 7 and 8).
- **Agreement:** section 9 compares the app's what-if reasons with SHAP for the same leads and reports how many of the
  top three details the two methods share.
- **Limitations:** the what-if method changes one detail at a time, so it cannot show details that only matter in
  combination. When a score is already very high or very low (for example 99% or 2%), changing one detail barely moves
  it, so the reasons shown are small even though the lead is clearly strong or weak. SHAP, which works in log-odds,
  does not have this problem, which is why it is used here as a check.